<!-- SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

# Multi-table PII Replacement

Run automatic PII discovery and replacement across the **entire CRM folder**, while preserving consistency through ordinary and Salesforce-style **polymorphic** FKs (`Task.WhoId` + `WhoType` → Contact or Lead).

#### What you'll learn

- Point `MultiTablePiiReplacer` at `/root/datasets/crm` and its `schema.yaml`
- Auto-discover one database-level replacement plan for every table
- Apply replacement to the whole folder with a shared runtime store
- Inspect polymorphic routing and the persisted plan/map artifacts

#### Notes

- Multi-table runs are **PII-only** — do not use `scope: database` with `SafeSynthesizer.run()`.
- Type discriminator columns (`WhoType`, …) are routing enums and are **not** replaced.
- `person.backend="managed"` loads a large locale parquet once and reuses it across tables; use `"faker"` if you want a lighter tutorial run.

```bash
pip install 'nemo-safe-synthesizer[notebook]'
```


## 1. Dataset and schema

CSVs and schema live under `/root/datasets/crm`. Filename stems must match schema table names.


In [1]:
from pathlib import Path

import pandas as pd

CRM = Path("/root/datasets/crm")
INPUT = CRM
SCHEMA_PATH = CRM / "schema.yaml"
# Parent for per-run workdirs (timestamped subfolders are created at transform time).
ARTIFACTS_ROOT = Path("multi_table_crm_artifacts")

assert SCHEMA_PATH.is_file(), SCHEMA_PATH
csv_files = sorted(INPUT.glob("*.csv"))
print(f"Found {len(csv_files)} tables in {INPUT}")
print("First tables:", [p.stem for p in csv_files[:8]])
pd.read_csv(INPUT / "Contact.csv").head(3)

Found 33 tables in /root/datasets/crm
First tables: ['Account', 'AccountContactRelation', 'AccountTeamMember', 'Asset', 'Campaign', 'CampaignMember', 'Case', 'CaseComment']


,Id,FirstName,LastName,Name,Title,Department,Email,Phone,MailingStreet,MailingCity,MailingState,MailingPostalCode,MailingCountry,LeadSource,CreatedDate,HasOptedOutOfEmail,AccountId,ReportsToId
0,003JB8s9DFpAbonYYC,Jorin W.,Rellor-Dell,Jorin W. Rellor-Dell,Account Director,Customer Success,jorin-w.rellor-dell.0-893@example.invalid,+1-971-555-0151,6822 Lantern Avenue,Northhaven,WA,90340,United States,Outbound,2025-06-30T21:23:06Z,0,001T9qvbtoySrxzIAC,NaN
1,003Z8JsUjU4pjJTIVY,Ivara N.,Belnor-Glen,Ivara N. Belnor-Glen,Account Director,Procurement,ivara-n.belnor-glen.1-5717@example.invalid,+1-415-555-0101,666 Orchard Road,Northhaven,OR,52036,United States,Web,2022-05-29T22:00:04Z,0,001nr8859QAKGpyAQH,NaN
2,003NEjTVlTQWPWbYWP,Fenn E.,Denvik-Isle,Fenn E. Denvik-Isle,Product Manager,Procurement,fenn-e.denvik-isle.2-8347@example.invalid,+1-415-555-0195,1046 Lantern Avenue,Willow Harbor,TX,41038,United States,Event,2026-03-20T22:40:16Z,0,001Lg7sq2NVLJLtIQP,NaN


## 2. Configure automatic discovery

`replacement_plan="auto_discovery"` examines every CSV with the schema and emits one auditable database-level plan. It discovers ordinary key domains and polymorphic routers; no hand-written plan is required.


In [2]:
from nemo_safe_synthesizer.config.replace_pii import ReplacePiiConfig

cfg = ReplacePiiConfig(
    schema_path=str(SCHEMA_PATH),
    replacement_plan="auto_discovery",
    replacement={"seed": 42, "locale": "en_US"},
    person={"backend": "faker"}, # faker or managed
)
cfg

ReplacePiiConfig(schema_version=1, llm_enhancement=False, replacement_plan='auto_discovery', llm=LLMConfig(model_provider=None, max_workers=64), replacement=PiiReplacementSettings(locale='en_US', seed=42), person=PiiPersonConfig(backend=<PiiPersonBackend.faker: 'faker'>, sdg_pgms_src=None, managed_assets_path=None), schema_path='/root/datasets/crm/schema.yaml')

## 3. Discover and replace the whole folder

This loads every top-level CSV, discovers a single database plan, processes tables in FK order, and writes one transformed CSV per input table.

Artifacts go under `multi_table_crm_artifacts/<timestamp>/` (plan + map at the run root, transformed CSVs in `output/`). Each re-run gets a new timestamped folder.


In [3]:
# Enable logging. Since we are only running one step of the pipeline, logging is not automatically enabled.
import os
os.environ["NSS_LOG_LEVEL"] = "WARNING"

from nemo_safe_synthesizer.observability import initialize_observability
initialize_observability()

In [4]:
from datetime import datetime

from nemo_safe_synthesizer.pii_replacer import MultiTablePiiReplacer

# Multi-table is PII-only (no full pipeline Workdir), so make a timestamped run folder ourselves.
WORKDIR = ARTIFACTS_ROOT / datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
OUTPUT = WORKDIR / "output"

replacer = MultiTablePiiReplacer(cfg, workdir=WORKDIR)
tables = replacer.transform_folder(INPUT, output_dir=OUTPUT)

print("Transformed tables:", len(tables))
print("Run workdir:", WORKDIR)
print("Plan artifact:", WORKDIR / "pii_replacement_plan.yaml")
print("Map artifact:", WORKDIR / "pii_replacement_map.yaml")
tables["Contact"][["Id", "FirstName", "LastName", "Email"]].head(3)

2026-08-13T20:23:28.764 | Nemo Safe Synthesizer |  user    |  warning |  persona_grouping.py: detect_structured_columns: 356 
[PII Replacement] Column 'Account.Phone' looks like phone_number by name but values do not look like phone numbers; skipped.  
2026-08-13T20:23:29.745 | Nemo Safe Synthesizer |  user    |  warning |  persona_grouping.py: detect_structured_columns: 374 
[PII Replacement] Column 'Asset.Name' values look like organizations, not people; skipped.  
2026-08-13T20:23:32.203 | Nemo Safe Synthesizer |  user    |  warning |  persona_grouping.py: detect_structured_columns: 356 
[PII Replacement] Column 'Contact.HasOptedOutOfEmail' looks like email by name but values do not match that entity; skipped.  
2026-08-13T20:23:34.604 | Nemo Safe Synthesizer |  user    |  warning |  persona_grouping.py: detect_structured_columns: 374 
[PII Replacement] Column 'EmailMessage.ToAddress' values lack house numbers (street name only); skipped.  
2026-08-13T20:23:38.713 | Nemo Safe Synthe

,Id,FirstName,LastName,Email
0,003eWhmT5dPzHuRQTV,Marie,Perry,marie.perry.4-534@example.invalid
1,003PoNXhoOeFYSSY1U,Juan,Vega,juan.vega.6-7122@example.invalid
2,003NOqblprQXyXlIYB,Christine,Parrish,christine.parrish.2-8368@example.invalid


## 4. Check polymorphic WhoId routing

`WhoType=Contact` rows reuse the `Contact.Id` domain map; `WhoType=Lead` uses `Lead.Id`. `WhoType` itself is unchanged.


In [5]:
contact_in = pd.read_csv(INPUT / "Contact.csv")
lead_in = pd.read_csv(INPUT / "Lead.csv")
task_in = pd.read_csv(INPUT / "Task.csv")
contact_out = tables["Contact"]
lead_out = tables["Lead"]
task_out = tables["Task"]

contact_map = dict(zip(contact_in["Id"], contact_out["Id"], strict=True))
lead_map = dict(zip(lead_in["Id"], lead_out["Id"], strict=True))

sample = task_in.head(8).copy()
sample["WhoId_out"] = task_out.loc[sample.index, "WhoId"].values
sample["WhoType_out"] = task_out.loc[sample.index, "WhoType"].values

ok_contact = [
    (t == "Contact" and wid in contact_map and contact_map[wid] == syn)
    or t != "Contact"
    for t, wid, syn in zip(sample["WhoType"], sample["WhoId"], sample["WhoId_out"], strict=True)
]
ok_lead = [
    (t == "Lead" and wid in lead_map and lead_map[wid] == syn) or t != "Lead"
    for t, wid, syn in zip(sample["WhoType"], sample["WhoId"], sample["WhoId_out"], strict=True)
]
print("WhoType unchanged:", list(sample["WhoType"]) == list(sample["WhoType_out"]))
print("Contact-typed WhoIds match Contact.Id map:", all(ok_contact))
print("Lead-typed WhoIds match Lead.Id map:", all(ok_lead))
sample[["WhoId", "WhoType", "WhoId_out", "WhoType_out", "Subject"]]

WhoType unchanged: True
Contact-typed WhoIds match Contact.Id map: True
Lead-typed WhoIds match Lead.Id map: True


,WhoId,WhoType,WhoId_out,WhoType_out,Subject
0,003pIbrFKNYOTCYQ25,Contact,003VeMNXOn4b6m3YPM,Contact,Review requirements
1,00QPaPEXwl1RZQQMH4,Lead,00Qah6FYwpEhRIU2ET,Lead,Follow up
2,003dFbt9bEr7AX1QQM,Contact,003XLYg95CtXxfyIBH,Contact,Send workshop summary
3,NaN,NaN,NaN,NaN,Review requirements
4,003CccN8Twj3aEsIKI,Contact,003y1knBdIDifuNYM6,Contact,Review requirements
5,0039CRawED7qCTrQZM,Contact,003GweaslhRRyCLI5I,Contact,Review requirements
6,003KYasN7G2ZFTXYU4,Contact,003ilgtGMtANeRHA6L,Contact,Follow up
7,003wQB1neemak4MQBQ,Contact,003q4mM8M5swTliYEV,Contact,Review requirements


## 5. Run from an audited plan

After reviewing/editing the discovered plan, point `replacement_plan` at the audited YAML to skip auto-discovery and apply that plan across the folder.


In [6]:
# Skip discovery: apply the audited database-scope plan.
AUDITED_PLAN = Path(
    "safe-synthesizer-artifacts/2026-08-13T16:22/pii_replacement_plan_audited.yaml"
)
assert AUDITED_PLAN.is_file(), AUDITED_PLAN

cfg_audited = ReplacePiiConfig(
    schema_path=str(SCHEMA_PATH),
    replacement_plan=str(AUDITED_PLAN),
    replacement={"seed": 42, "locale": "en_US"},
    person={"backend": "faker"},  # faker or managed
)

WORKDIR_AUDITED = ARTIFACTS_ROOT / f"{datetime.now().strftime('%Y-%m-%dT%H:%M:%S')}_from_audited"
OUTPUT_AUDITED = WORKDIR_AUDITED / "output"

replacer_audited = MultiTablePiiReplacer(cfg_audited, workdir=WORKDIR_AUDITED)
tables_audited = replacer_audited.transform_folder(INPUT, output_dir=OUTPUT_AUDITED)

print("Transformed tables:", len(tables_audited))
print("Run workdir:", WORKDIR_AUDITED)
print("Plan used:", AUDITED_PLAN.resolve())
print("Plan artifact:", WORKDIR_AUDITED / "pii_replacement_plan.yaml")
print("Map artifact:", WORKDIR_AUDITED / "pii_replacement_map.yaml")
tables_audited["Contact"][["Id", "FirstName", "LastName", "Email"]].head(3)


2026-08-13T21:06:59.610 | Nemo Safe Synthesizer |  user    |  warning |  order.py: processing_order: 56 
[PII Replacement] Schema FK graph has a cycle involving CampaignMember, EmailMessage, Event, EventRelation, Lead, ObjectTerritory2Association, Opportunity, OpportunityContactRole, OpportunityLineItem, OpportunityTeamMember, Order, OrderItem, Partner, Quote, QuoteLineItem, Task, TaskRelation; falling back to schema listing order for those tables.  
Transformed tables: 33
Run workdir: multi_table_crm_artifacts/2026-08-13T21:06:59_from_audited
Plan used: /root/Safe-Synthesizer-multi-table-pii-repl/docs/tutorials/safe-synthesizer-artifacts/2026-08-13T16:22/pii_replacement_plan_audited.yaml
Plan artifact: multi_table_crm_artifacts/2026-08-13T21:06:59_from_audited/pii_replacement_plan.yaml
Map artifact: multi_table_crm_artifacts/2026-08-13T21:06:59_from_audited/pii_replacement_map.yaml


,Id,FirstName,LastName,Email
0,003eWhmT5dPzHuRQTV,Marie,Perry,marie.perry.4-534@example.invalid
1,003PoNXhoOeFYSSY1U,Juan,Vega,juan.vega.6-7122@example.invalid
2,003NOqblprQXyXlIYB,Christine,Parrish,christine.parrish.2-8368@example.invalid
